# Train the Bibley models, part 2 of 2 (1B+ models)

Settings → Accelerator → **GPU T4**, Internet **on**. Then Save Version → **Save & Run All** (runs in the background, up to 12 h). Finished GGUF files land in the Output tab.

In [ ]:
!git clone -b bibley-models https://github.com/olivercraftboycompany-design/bibley-ai.git
%cd bibley-ai
!pip install -q -r training/requirements-train.txt

In [ ]:
# Build the KJV training set (about 15,200 examples)
!python training/build_dataset.py

In [ ]:
# Models trained in this notebook (names from training/models.json)
MODELS = [
    "Bibley-TinyLlama-1.1B",
    "Bibley-Granite-H-1B",
    "Bibley-OLMo2-1B"
]
for m in MODELS:
    !python training/train_lora.py --model {m}

In [ ]:
# How often does each model quote the verse exactly?
for m in MODELS:
    !python training/eval_quotes.py training/out/{m}/merged --n 100

In [ ]:
# Shrink each model under 1 GB as GGUF, keep only the GGUF to save output space
import shutil
for m in MODELS:
    !bash training/export_gguf.sh {m}
    shutil.rmtree(f'training/out/{m}/merged', ignore_errors=True)
shutil.rmtree('training/llama.cpp', ignore_errors=True)
shutil.rmtree('training/data', ignore_errors=True)

In [ ]:
# Optional: upload to Hugging Face if a secret named HF_TOKEN exists (Add-ons → Secrets). Skipped otherwise.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
    user = api.whoami()['name']
    for m in MODELS:
        repo = f'{user}/{m}'
        api.create_repo(repo, exist_ok=True)
        api.upload_folder(folder_path=f'training/out/{m}/gguf', repo_id=repo)
        print('uploaded', repo)
except Exception as e:
    print('Hugging Face upload skipped:', e)